# Stage 5B — Blind Label-Free Target Scoring and Prediction Freeze

**Protocol:** Prospective Retinal Blind Test v0.1  
**Authorised edges:** DeepDRiD → APTOS_2019; DeepDRiD → IDRiD  
**Status before first execution:** executable specification fixed; target diagnostic labels remain sealed  
**Execution device:** CPU for deterministic continuity with Stage 5A

Run the notebook from top to bottom. The first code cell verifies upstream commitments and writes a machine-readable pre-registration seal **before any target image is opened**. Later cells are resumable from fixed-width Unicode and float32 NumPy checkpoints. The notebook blocks every attempted file open under `99_Label_Quarantine_DO_NOT_OPEN_BEFORE_STAGE6`.

This stage freezes label-free component scores, ordinal risk statements, uncertainty, abstention and intervention recommendations. It does **not** read target labels, calculate target performance, refit the source axis, or validate a general transfer predictor from two edges.


In [1]:
#@title 05B-0. Preflight, integrity commitments, and pre-image-access protocol seal

from pathlib import Path
from datetime import datetime, timezone
from io import BytesIO
from PIL import Image, ImageFile, ImageFilter, ImageEnhance, ImageDraw

import hashlib
import json
import os
import platform
import random
import re
import sys
import warnings

import numpy as np
import pandas as pd
import sklearn
import torch
import torchvision
import PIL


# ============================================================
# 0. Locked execution controls
# ============================================================

RUN_STAGE5B = True
RANDOM_SEED = 20260721
EXECUTION_DEVICE = "cpu"
FROZEN_FEATURE_DIMENSION = 2048
FROZEN_BACKBONE = "ResNet50_IMAGENET1K_V2"
EMBEDDING_NORMALISATION = "L2"
ENDPOINT_ID = "MODERATE_OR_WORSE_DR_GRADE_GE_2"
BATCH_SIZE = 16
NUM_WORKERS = 0
CHECKPOINT_EVERY_BATCHES = 10
TRANSFORM_ANCHORS_PER_TARGET = 64
SOURCE_ANCHOR_EYES_PER_CLASS = 32
N_BOOTSTRAP = 500
K_NEIGHBOURS = 5
MMD_MAX_RECORDS = 512
OT_RECORDS = 128

assert RUN_STAGE5B, "Set RUN_STAGE5B=True only when ready to create the seal and score targets."
assert EXECUTION_DEVICE == "cpu"

os.environ["PYTHONHASHSEED"] = str(RANDOM_SEED)
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
torch.set_num_threads(max(1, min(8, os.cpu_count() or 2)))
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
ImageFile.LOAD_TRUNCATED_IMAGES = False


# ============================================================
# 1. Resolve Drive and frozen project paths
# ============================================================

if Path("/content/drive/MyDrive").is_dir():
    DRIVE_ROOT = Path("/content/drive/MyDrive")
elif Path("/content/gdrive/MyDrive").is_dir():
    DRIVE_ROOT = Path("/content/gdrive/MyDrive")
else:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_ROOT = Path("/content/drive/MyDrive")

PROJECT_ROOT = DRIVE_ROOT / "Cross-Modal_Diagnostic_Observability"
NOTEBOOK_PATH = (
    PROJECT_ROOT / "05_Code" / "Retinal_DR" /
    "Retinal_DR_Stage5B_Blind_LabelFree_Scoring_And_Prediction_Freeze_v0.1.ipynb"
)
TEST_ROOT = (
    PROJECT_ROOT / "06_Data_Records" / "Retinal_DR" /
    "Prospective_Retinal_Blind_Test_v0.1"
)
STAGE2_ROOT = TEST_ROOT / "Stage2_Acquisition_And_Quarantine_v0.1"
SOURCE_FINAL_ROOT = (
    TEST_ROOT / "Stage4_Source_Canonicalisation_v0.1" /
    "07_Source_Finalisation"
)
STAGE5A_ROOT = TEST_ROOT / "Stage5_Source_Recoverability_And_Axis_Freeze_v0.1"
STAGE5B_ROOT = TEST_ROOT / "Stage5B_Label_Free_Target_Scoring_And_Prediction_Freeze_v0.1"
PROTOCOL_ROOT = STAGE5B_ROOT / "00_Protocol"
SCORE_ROOT = STAGE5B_ROOT / "01_Label_Free_Scores"
FREEZE_ROOT = STAGE5B_ROOT / "02_Prediction_Freeze"
RESULT_ROOT = STAGE5B_ROOT / "03_Results"

for directory in [PROTOCOL_ROOT, SCORE_ROOT, FREEZE_ROOT, RESULT_ROOT]:
    directory.mkdir(parents=True, exist_ok=True)

STAGE5A_DECISION_PATH = (
    STAGE5A_ROOT / "03_Results" /
    "Stage5A_Source_Recoverability_And_Axis_Freeze_Decision_v0.1.json"
)
SOURCE_MANIFEST_PATH = (
    SOURCE_FINAL_ROOT / "01_Final_Source_Manifests" /
    "Stage4G_Final_Canonical_Source_Manifest_v0.1.csv"
)
SOURCE_EMBEDDING_ROOT = STAGE5A_ROOT / "01_Frozen_Embeddings"
SOURCE_AXIS_ROOT = STAGE5A_ROOT / "02_Frozen_Source_Axes"
SOURCE_EMBEDDING_PATH = (
    SOURCE_EMBEDDING_ROOT /
    "DeepDRiD_Canonical_ResNet50V2_L2_Embeddings_v0.1.npy"
)
SOURCE_EMBEDDING_IDS_PATH = (
    SOURCE_EMBEDDING_ROOT /
    "DeepDRiD_Canonical_ResNet50V2_L2_ImageIDs_v0.1.npy"
)
SOURCE_EMBEDDING_METADATA_PATH = (
    SOURCE_EMBEDDING_ROOT /
    "DeepDRiD_Canonical_ResNet50V2_L2_Embeddings_Metadata_v0.1.json"
)
SOURCE_AXIS_PATH = SOURCE_AXIS_ROOT / "DeepDRiD_Frozen_Source_Axis_v0.1.npz"
SOURCE_AXIS_METADATA_PATH = (
    SOURCE_AXIS_ROOT / "DeepDRiD_Frozen_Source_Axis_Metadata_v0.1.json"
)

TARGET_MANIFEST_PATHS = {
    "APTOS_2019": (
        TEST_ROOT / "03_Sealed_Evaluation" /
        "APTOS_2019_Sealed_Evaluation_Manifest_v0.1.csv"
    ),
    "IDRiD": (
        TEST_ROOT / "03_Sealed_Evaluation" /
        "IDRiD_Sealed_Evaluation_Manifest_v0.1.csv"
    ),
}
EXPECTED_TARGET_COUNTS = {"APTOS_2019": 1080, "IDRiD": 102}
FORBIDDEN_LABEL_ROOT = (
    TEST_ROOT / "99_Label_Quarantine_DO_NOT_OPEN_BEFORE_STAGE6"
)
SEAL_PATH = PROTOCOL_ROOT / "Stage5B_Executable_PreRegistration_Seal_v0.1.json"
RUNTIME_STATE_PATH = RESULT_ROOT / "Stage5B_Runtime_Access_State_v0.1.json"
FINAL_FREEZE_PATH = FREEZE_ROOT / "Stage5B_Prediction_Freeze_Complete_v0.1.json"


# ============================================================
# 2. Stable hashing and atomic writers
# ============================================================

def utc_now():
    return datetime.now(timezone.utc).isoformat()


def sha256_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        while True:
            chunk = handle.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()


def sha256_json(value):
    payload = json.dumps(
        value, sort_keys=True, separators=(",", ":"), ensure_ascii=False
    ).encode("utf-8")
    return hashlib.sha256(payload).hexdigest()


def notebook_source_sha256(path):
    with Path(path).open("r", encoding="utf-8") as handle:
        notebook = json.load(handle)
    frozen = {
        "nbformat": notebook.get("nbformat"),
        "nbformat_minor": notebook.get("nbformat_minor"),
        "cells": [
            {
                "cell_type": cell.get("cell_type"),
                "source": "".join(cell.get("source", [])),
            }
            for cell in notebook.get("cells", [])
        ],
    }
    return sha256_json(frozen)


def atomic_json(path, payload):
    path = Path(path)
    temporary = path.with_suffix(path.suffix + ".tmp")
    with temporary.open("w", encoding="utf-8") as handle:
        json.dump(payload, handle, indent=2, ensure_ascii=False)
    os.replace(temporary, path)


def deterministic_rank(text, namespace=""):
    return hashlib.sha256(
        f"{RANDOM_SEED}|{namespace}|{text}".encode("utf-8")
    ).hexdigest()


# ============================================================
# 3. Exact executable specification frozen before target access
# ============================================================

TRANSFORMATION_SPEC = {
    "RESOLUTION_192": {
        "family": "resolution_frequency",
        "operation": "resize 768->192 bilinear, then 192->768 bicubic",
    },
    "GAUSSIAN_BLUR_R2": {
        "family": "blur_noise",
        "operation": "PIL GaussianBlur radius=2.0 pixels",
    },
    "GAUSSIAN_NOISE_S004": {
        "family": "blur_noise",
        "operation": "RGB Gaussian noise sigma=0.04 on [0,1]; per-image SHA256 seed",
    },
    "CONTRAST_065": {
        "family": "colour_contrast",
        "operation": "PIL ImageEnhance.Contrast factor=0.65 in RGB",
    },
    "SATURATION_050": {
        "family": "colour_contrast",
        "operation": "PIL ImageEnhance.Color factor=0.50 in RGB",
    },
    "CENTRAL_CROP_080": {
        "family": "field_of_view_occlusion",
        "operation": "retain central 80% square, resize to 768 bicubic",
    },
    "CENTRAL_OCCLUSION_R015": {
        "family": "field_of_view_occlusion",
        "operation": "central circular mask radius=15% image width; fill channel medians",
    },
    "MEDIAN_FILTER_5": {
        "family": "structure_texture",
        "operation": "PIL MedianFilter size=5",
    },
    "JPEG_Q40": {
        "family": "compression_acquisition",
        "operation": "JPEG quality=40, subsampling=2, optimize=False, progressive=False",
    },
}

COMPONENT_SPEC = {
    "R_s": "inherit sealed Stage5A DeepDRiD pass; verify hashes; never refit",
    "S_t_given_s": (
        "fraction of target L2 embeddings whose 5-NN cosine distance to DeepDRiD "
        "development eye embeddings is <= the source leave-one-eye-out 95th percentile"
    ),
    "E_s": (
        "per-transform median absolute frozen-axis logit change divided by the IQR "
        "of DeepDRiD validation eye logits"
    ),
    "evidence_compatibility": (
        "exp(-uniform mean over transforms of abs(log((target response+1e-6)/"
        "(source response+1e-6))))"
    ),
    "N_s_to_t": (
        "source-fitted PCA-32 domain-classifier five-fold AUC plus RBF-MMD and "
        "128-record optimal-assignment cosine cost"
    ),
    "U_s_to_t": (
        "500 deterministic target-record bootstraps for support fraction and evidence "
        "compatibility; source reference remains fixed"
    ),
}

BASELINE_SPEC = {
    "source_validation": "sealed Stage5A source validation performance, no target input",
    "confidence_entropy": "mean binary confidence and entropy of frozen-axis target probabilities",
    "ATC": (
        "source-validation confidence quantile at observed source validation error; "
        "target estimated accuracy is fraction at or above that fixed threshold"
    ),
    "MMD": "biased RBF-MMD^2 on deterministic equal samples; source median squared distance bandwidth",
    "OT": "mean cosine cost under scipy linear_sum_assignment on deterministic equal samples",
    "domain_classifier": "source-fitted PCA-32 plus balanced logistic regression; five-fold domain AUC",
    "transform_stability": "median normalized logit response and prediction-flip rate per transform",
}

DECISION_THRESHOLDS = {
    "abstain_if_support_fraction_below": 0.50,
    "abstain_if_fraction_beyond_source_q99_above": 0.50,
    "abstain_if_evidence_compatibility_below": 0.40,
    "high_risk_if_support_fraction_below": 0.70,
    "high_risk_if_evidence_compatibility_below": 0.55,
    "high_risk_if_domain_auc_at_or_above": 0.90,
    "moderate_risk_if_support_fraction_below": 0.90,
    "moderate_risk_if_evidence_compatibility_below": 0.75,
    "moderate_risk_if_domain_auc_at_or_above": 0.75,
    "bootstrap_interval": [0.025, 0.975],
    "relative_ordering": "not issued for n=2; ties/indeterminacy preserved",
}

OUTPUT_SCHEMA = {
    "edge_json": [
        "edge_id", "source", "target", "eligibility", "component_scores",
        "baselines", "uncertainty", "transfer_risk", "dominant_failure_mode",
        "minimal_intervention", "abstain", "abstention_reason_codes",
        "relative_statement", "narrative", "input_hashes", "output_created_utc",
    ],
    "edge_summary_csv": "one row per authorised edge; no target outcomes",
    "target_image_scores_csv": "image_id, frozen logit/probability, no labels",
    "transformation_response_csv": "source/target aggregate response by frozen transform",
    "decision_pdf": "human-readable pre-unseal sheet generated from frozen JSON",
    "integrity_manifest_csv": "SHA256 for every pre-unseal output before final freeze record",
}

required_paths = [
    NOTEBOOK_PATH, STAGE5A_DECISION_PATH, SOURCE_MANIFEST_PATH,
    SOURCE_EMBEDDING_PATH, SOURCE_EMBEDDING_IDS_PATH,
    SOURCE_EMBEDDING_METADATA_PATH, SOURCE_AXIS_PATH,
    SOURCE_AXIS_METADATA_PATH, *TARGET_MANIFEST_PATHS.values(),
]
for required_path in required_paths:
    assert required_path.is_file(), f"Missing required frozen artefact: {required_path}"

with STAGE5A_DECISION_PATH.open("r", encoding="utf-8") as handle:
    stage5a_decision = json.load(handle)
assert stage5a_decision["decision"] == "PARTIAL_PASS_RETAIN_ONLY_EDGES_FROM_RECOVERABLE_SOURCE"
authorised_pairs = {
    (str(edge["source"]), str(edge["target"]))
    for edge in stage5a_decision["authorised_edges"]
}
assert authorised_pairs == {
    ("DeepDRiD", "APTOS_2019"), ("DeepDRiD", "IDRiD")
}

target_manifests = {}
target_manifest_hashes = {}
forbidden_column_fragments = ("label", "grade", "diagnosis", "moderate_or_worse")
for target_name, manifest_path in TARGET_MANIFEST_PATHS.items():
    frame = pd.read_csv(manifest_path)
    assert len(frame) == EXPECTED_TARGET_COUNTS[target_name]
    assert frame["dataset"].astype(str).eq(target_name).all()
    assert frame["image_id"].astype(str).is_unique
    assert not any(
        fragment in str(column).lower()
        for column in frame.columns
        for fragment in forbidden_column_fragments
    )
    assert set(["dataset", "image_id", "canonical_image_path", "canonical_sha256"]).issubset(frame.columns)
    target_manifests[target_name] = frame.sort_values("image_id").reset_index(drop=True)
    target_manifest_hashes[target_name] = sha256_file(manifest_path)

environment = {
    "python": sys.version,
    "platform": platform.platform(),
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "torch": torch.__version__,
    "torchvision": torchvision.__version__,
    "sklearn": sklearn.__version__,
    "pillow": PIL.__version__,
    "execution_device": EXECUTION_DEVICE,
    "cpu_threads": torch.get_num_threads(),
}

input_hashes = {
    "notebook_source_sha256": notebook_source_sha256(NOTEBOOK_PATH),
    "stage5a_decision_sha256": sha256_file(STAGE5A_DECISION_PATH),
    "source_manifest_sha256": sha256_file(SOURCE_MANIFEST_PATH),
    "source_embedding_sha256": sha256_file(SOURCE_EMBEDDING_PATH),
    "source_embedding_ids_sha256": sha256_file(SOURCE_EMBEDDING_IDS_PATH),
    "source_axis_npz_sha256": sha256_file(SOURCE_AXIS_PATH),
    "source_axis_metadata_sha256": sha256_file(SOURCE_AXIS_METADATA_PATH),
    "aptos_manifest_sha256": target_manifest_hashes["APTOS_2019"],
    "idrid_manifest_sha256": target_manifest_hashes["IDRiD"],
    "transformation_spec_sha256": sha256_json(TRANSFORMATION_SPEC),
    "component_spec_sha256": sha256_json(COMPONENT_SPEC),
    "baseline_spec_sha256": sha256_json(BASELINE_SPEC),
    "decision_thresholds_sha256": sha256_json(DECISION_THRESHOLDS),
    "output_schema_sha256": sha256_json(OUTPUT_SCHEMA),
    "environment_lock_sha256": sha256_json(environment),
}

seal_payload = {
    "stage": "Stage5B",
    "version": "v0.1",
    "status": "SEALED_BEFORE_FIRST_TARGET_IMAGE_ACCESS",
    "sealed_utc": utc_now(),
    "endpoint_id": ENDPOINT_ID,
    "authorised_edges": [
        {"edge_id": "DDR_APTOS_v0.1", "source": "DeepDRiD", "target": "APTOS_2019"},
        {"edge_id": "DDR_IDRiD_v0.1", "source": "DeepDRiD", "target": "IDRiD"},
    ],
    "retired_source": "EyePACS_2015",
    "target_images_accessed_at_seal": False,
    "target_sealed_label_files_accessed_at_seal": False,
    "target_performance_observed_at_seal": False,
    "transformation_spec": TRANSFORMATION_SPEC,
    "component_spec": COMPONENT_SPEC,
    "baseline_spec": BASELINE_SPEC,
    "decision_thresholds": DECISION_THRESHOLDS,
    "output_schema": OUTPUT_SCHEMA,
    "environment": environment,
    "input_hashes": input_hashes,
    "review_status": (
        "EXECUTABLE_SPECIFICATION_HASHED_BEFORE_TARGET_IMAGE_ACCESS; "
        "INDEPENDENT_CUSTODIAN_REVIEW_NOT_CLAIMED"
    ),
    "scientific_scope": (
        "Prospective two-edge feasibility/falsification only; no two-edge calibration, "
        "general predictor claim, or post-outcome tuning."
    ),
}
seal_payload["seal_sha256"] = sha256_json(seal_payload)

if SEAL_PATH.is_file():
    with SEAL_PATH.open("r", encoding="utf-8") as handle:
        existing_seal = json.load(handle)
    for key, value in input_hashes.items():
        assert existing_seal["input_hashes"][key] == value, (
            f"Existing Stage5B seal no longer matches {key}. Stop without scoring."
        )
    seal_payload = existing_seal
    print("Existing Stage 5B pre-image-access seal verified; resumable execution authorised.")
else:
    assert not FINAL_FREEZE_PATH.exists()
    atomic_json(SEAL_PATH, seal_payload)
    assert sha256_file(SEAL_PATH)
    print("New Stage 5B executable pre-registration seal written before target image access.")

runtime_state = {
    "stage": "Stage5B",
    "seal_path": str(SEAL_PATH),
    "seal_sha256": seal_payload["seal_sha256"],
    "target_images_accessed": False,
    "target_sealed_label_files_accessed": False,
    "target_performance_observed": False,
    "source_target_transfer_performance_observed": False,
    "last_updated_utc": utc_now(),
}
if RUNTIME_STATE_PATH.is_file():
    with RUNTIME_STATE_PATH.open("r", encoding="utf-8") as handle:
        prior_state = json.load(handle)
    if prior_state.get("target_images_accessed"):
        runtime_state["target_images_accessed"] = True
        runtime_state["first_target_image_access_utc"] = prior_state.get("first_target_image_access_utc")
atomic_json(RUNTIME_STATE_PATH, runtime_state)

# Permanent runtime guard: any attempted file open under the label quarantine aborts.
forbidden_prefix = str(FORBIDDEN_LABEL_ROOT.resolve())
def _stage5b_label_open_guard(event, args):
    if event != "open" or not args:
        return
    candidate = args[0]
    if isinstance(candidate, (str, os.PathLike)):
        try:
            resolved = str(Path(candidate).resolve())
        except Exception:
            return
        if resolved == forbidden_prefix or resolved.startswith(forbidden_prefix + os.sep):
            raise PermissionError(
                "Stage 5B blocked attempted access to target label quarantine: " + resolved
            )
sys.addaudithook(_stage5b_label_open_guard)

print("\n================ STAGE 5B PREFLIGHT SEALED ================")
print("Notebook source hash:", input_hashes["notebook_source_sha256"])
print("Protocol seal hash:", seal_payload["seal_sha256"])
print("Authorised edges:", sorted(authorised_pairs))
print("Target images accessed:", runtime_state["target_images_accessed"])
print("Target sealed-label files accessed: False")
print("Target performance observed: False")
display(pd.DataFrame([
    {"target": name, "label_free_images": len(frame), "manifest_sha256": target_manifest_hashes[name]}
    for name, frame in target_manifests.items()
]))


Mounted at /content/drive
New Stage 5B executable pre-registration seal written before target image access.

================ STAGE 5B PREFLIGHT SEALED ================
Notebook source hash: b9bcaf1a90844a1e5bdb333d6fb1abc4fcafeca34778d9ae0601a38b43150b1a
Protocol seal hash: 5d30003fabfcc37f1909399441a1c6061e6b94666f554b500c1a3e31760a7649
Authorised edges: [('DeepDRiD', 'APTOS_2019'), ('DeepDRiD', 'IDRiD')]
Target images accessed: False
Target sealed-label files accessed: False
Target performance observed: False


,target,label_free_images,manifest_sha256
0,APTOS_2019,1080,cacde3016911859c68623608023d818a349de314209771...
1,IDRiD,102,4b2998804d450515ed079071f028a935176fcad8292dfe...


In [2]:
#@title 05B-1. Load frozen source assets and extract resumable target embeddings

from torch.utils.data import Dataset, DataLoader
from torchvision.models import resnet50, ResNet50_Weights
import torch.nn as nn
import torch.nn.functional as F
from tqdm.auto import tqdm


# ============================================================
# 1. Frozen model and source axis
# ============================================================

device = torch.device("cpu")
weights = ResNet50_Weights.IMAGENET1K_V2
inference_transform = weights.transforms()
model = resnet50(weights=weights)
model.fc = nn.Identity()
model.eval()
for parameter in model.parameters():
    parameter.requires_grad = False
model = model.to(device)

axis = np.load(SOURCE_AXIS_PATH, allow_pickle=False)
axis_coefficient_raw = axis["coefficient_raw"].astype(np.float64)
axis_intercept_raw = float(axis["intercept_raw"][0])
axis_coefficient_standardised = axis["coefficient_standardised"].astype(np.float64)
axis_intercept_standardised = float(axis["intercept_standardised"][0])
axis_scaler_mean = axis["scaler_mean"].astype(np.float64)
axis_scaler_scale = axis["scaler_scale"].astype(np.float64)
assert axis_coefficient_raw.shape == (FROZEN_FEATURE_DIMENSION,)


def sigmoid(values):
    values = np.asarray(values, dtype=np.float64)
    return np.where(
        values >= 0,
        1.0 / (1.0 + np.exp(-values)),
        np.exp(values) / (1.0 + np.exp(values)),
    )


def raw_axis_logit(embeddings):
    return np.asarray(embeddings, dtype=np.float64) @ axis_coefficient_raw + axis_intercept_raw


# ============================================================
# 2. Source rows, embeddings, eye units and numerical equivalence
# ============================================================

def normalise_eye_side(value):
    if pd.isna(value):
        return ""
    key = re.sub(r"[^a-z0-9]+", "", str(value).lower())
    if key in {"left", "l", "lefteye", "os"}:
        return "left"
    if key in {"right", "r", "righteye", "od"}:
        return "right"
    return ""


def eye_side_from_image_id(image_id):
    text = str(image_id).lower()
    if any(re.search(pattern, text) for pattern in [r"(?:^|[_\-\s])left(?:$|[_\-\s])", r"(?:^|[_\-\s])l(?:$|[_\-\s])"]):
        return "left"
    if any(re.search(pattern, text) for pattern in [r"(?:^|[_\-\s])right(?:$|[_\-\s])", r"(?:^|[_\-\s])r(?:$|[_\-\s])"]):
        return "right"
    return ""


def build_eye_unit_id(row):
    side = normalise_eye_side(row["eye_side"])
    if not side:
        side = eye_side_from_image_id(row["image_id"])
    if side:
        return f"{row['dataset']}__PATIENT_{row['patient_id']}__EYE_{side}"
    return f"{row['dataset']}__IMAGE_{row['image_id']}"


source_manifest = pd.read_csv(SOURCE_MANIFEST_PATH)
source_manifest = (
    source_manifest[source_manifest["dataset"].astype(str).eq("DeepDRiD")]
    .sort_values(["source_partition", "image_id"])
    .reset_index(drop=True)
    .copy()
)
assert len(source_manifest) == 1600
source_manifest["eye_unit_id"] = source_manifest.apply(build_eye_unit_id, axis=1)
source_ids = np.load(SOURCE_EMBEDDING_IDS_PATH, allow_pickle=False).astype(str)
source_embeddings = np.load(SOURCE_EMBEDDING_PATH, mmap_mode="r", allow_pickle=False)
assert source_embeddings.shape == (1600, FROZEN_FEATURE_DIMENSION)
assert np.array_equal(source_ids, source_manifest["image_id"].astype(str).to_numpy())
assert np.allclose(np.linalg.norm(source_embeddings, axis=1), 1.0, atol=1e-5)

source_image_logits = raw_axis_logit(source_embeddings)
standardised_logits = (
    (np.asarray(source_embeddings, dtype=np.float64) - axis_scaler_mean) /
    axis_scaler_scale
) @ axis_coefficient_standardised + axis_intercept_standardised
maximum_axis_equivalence_error = float(np.max(np.abs(source_image_logits - standardised_logits)))
assert maximum_axis_equivalence_error < 1e-10
source_manifest["axis_logit"] = source_image_logits
source_manifest["probability"] = sigmoid(source_image_logits)

def aggregate_source_eyes(frame, embeddings):
    records = []
    for eye_unit_id, indices in frame.groupby("eye_unit_id", sort=True).indices.items():
        indices = np.asarray(indices, dtype=np.int64)
        embedding = np.asarray(embeddings[indices], dtype=np.float64).mean(axis=0)
        norm = float(np.linalg.norm(embedding))
        assert norm > 0
        embedding /= norm
        rows = frame.iloc[indices]
        labels = rows["moderate_or_worse_dr"].astype(int).unique()
        assert len(labels) == 1
        records.append({
            "eye_unit_id": str(eye_unit_id),
            "patient_id": str(rows["patient_id"].iloc[0]),
            "source_partition": str(rows["source_partition"].iloc[0]),
            "label": int(labels[0]),
            "probability": float(rows["probability"].mean()),
            "axis_logit": float(rows["axis_logit"].mean()),
            "embedding": embedding,
            "images": int(len(rows)),
        })
    table = pd.DataFrame([{key: value for key, value in row.items() if key != "embedding"} for row in records])
    matrix = np.stack([row["embedding"] for row in records]).astype(np.float32)
    return table, matrix

source_eye_table, source_eye_embeddings = aggregate_source_eyes(source_manifest, source_embeddings)
assert len(source_eye_table) == 800
source_development_mask = source_eye_table["source_partition"].eq("development").to_numpy()
source_validation_mask = source_eye_table["source_partition"].eq("validation").to_numpy()
assert source_development_mask.sum() == 600
assert source_validation_mask.sum() == 200


# ============================================================
# 3. Target embedding checkpoints; image bytes are opened only now
# ============================================================

class CanonicalTargetDataset(Dataset):
    def __init__(self, dataframe, indices):
        self.dataframe = dataframe.reset_index(drop=True)
        self.indices = np.asarray(indices, dtype=np.int64)

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, local_index):
        global_index = int(self.indices[local_index])
        row = self.dataframe.iloc[global_index]
        image_path = Path(row["canonical_image_path"])
        image_bytes = image_path.read_bytes()
        observed_hash = hashlib.sha256(image_bytes).hexdigest()
        assert observed_hash == str(row["canonical_sha256"])
        with Image.open(BytesIO(image_bytes)) as image:
            image.load()
            assert image.size == (768, 768)
            tensor = inference_transform(image.convert("RGB"))
        return tensor, global_index


def target_manifest_commitment(frame):
    rows = frame[["dataset", "image_id", "canonical_sha256"]].astype(str).to_dict("records")
    return sha256_json(rows)


def load_or_extract_target_embeddings(target_name, frame):
    safe_name = target_name.replace(" ", "_")
    embedding_path = SCORE_ROOT / f"{safe_name}_Canonical_ResNet50V2_L2_Embeddings_v0.1.npy"
    completion_path = SCORE_ROOT / f"{safe_name}_Canonical_ResNet50V2_L2_Completion_v0.1.npy"
    image_id_path = SCORE_ROOT / f"{safe_name}_Canonical_ResNet50V2_L2_ImageIDs_v0.1.npy"
    metadata_path = SCORE_ROOT / f"{safe_name}_Canonical_ResNet50V2_L2_Embeddings_Metadata_v0.1.json"
    commitment = target_manifest_commitment(frame)
    expected_shape = (len(frame), FROZEN_FEATURE_DIMENSION)

    resume_ok = False
    if all(path.is_file() for path in [embedding_path, completion_path, image_id_path, metadata_path]):
        try:
            with metadata_path.open("r", encoding="utf-8") as handle:
                metadata = json.load(handle)
            stored_ids = np.load(image_id_path, allow_pickle=False).astype(str)
            completion = np.load(completion_path, allow_pickle=False).astype(bool)
            stored = np.load(embedding_path, mmap_mode="r", allow_pickle=False)
            resume_ok = bool(
                metadata["manifest_commitment_sha256"] == commitment and
                stored.shape == expected_shape and
                completion.shape == (len(frame),) and
                np.array_equal(stored_ids, frame["image_id"].astype(str).to_numpy())
            )
            del stored
        except Exception:
            resume_ok = False

    if not resume_ok:
        for path in [embedding_path, completion_path, image_id_path, metadata_path]:
            if path.exists():
                path.unlink()
        memmap = np.lib.format.open_memmap(
            embedding_path, mode="w+", dtype=np.float32, shape=expected_shape
        )
        memmap[:] = 0.0
        memmap.flush()
        del memmap
        completion = np.zeros(len(frame), dtype=bool)
        np.save(completion_path, completion, allow_pickle=False)
        max_length = max(1, int(frame["image_id"].astype(str).str.len().max()))
        fixed_ids = frame["image_id"].astype(str).to_numpy(dtype=f"<U{max_length}")
        np.save(image_id_path, fixed_ids, allow_pickle=False)
        atomic_json(metadata_path, {
            "dataset": target_name,
            "status": "IN_PROGRESS",
            "images": len(frame),
            "completed_images": 0,
            "embedding_shape": list(expected_shape),
            "dtype": "float32",
            "backbone": FROZEN_BACKBONE,
            "embedding_normalisation": EMBEDDING_NORMALISATION,
            "manifest_commitment_sha256": commitment,
            "execution_device": EXECUTION_DEVICE,
            "protocol_seal_sha256": seal_payload["seal_sha256"],
            "target_sealed_labels_accessed": False,
            "target_performance_observed": False,
        })

    completion = np.load(completion_path, allow_pickle=False).astype(bool)
    missing_indices = np.flatnonzero(~completion)
    print(f"{target_name}: completed embeddings {completion.sum()}/{len(completion)}")

    if len(missing_indices):
        if not runtime_state.get("target_images_accessed"):
            runtime_state["target_images_accessed"] = True
            runtime_state["first_target_image_access_utc"] = utc_now()
            runtime_state["last_updated_utc"] = utc_now()
            atomic_json(RUNTIME_STATE_PATH, runtime_state)

        memmap = np.lib.format.open_memmap(
            embedding_path, mode="r+", dtype=np.float32, shape=expected_shape
        )
        dataset = CanonicalTargetDataset(frame, missing_indices)
        loader = DataLoader(
            dataset, batch_size=BATCH_SIZE, shuffle=False,
            num_workers=NUM_WORKERS, pin_memory=False, drop_last=False,
        )
        with torch.inference_mode():
            for batch_number, (image_batch, global_indices) in enumerate(
                tqdm(loader, desc=f"CPU extracting {target_name} target embeddings"), start=1
            ):
                features = F.normalize(model(image_batch.to(device)), p=2, dim=1)
                indices = global_indices.numpy().astype(np.int64)
                memmap[indices] = features.cpu().numpy().astype(np.float32)
                completion[indices] = True
                if batch_number % CHECKPOINT_EVERY_BATCHES == 0:
                    memmap.flush()
                    np.save(completion_path, completion, allow_pickle=False)
                    with metadata_path.open("r", encoding="utf-8") as handle:
                        metadata = json.load(handle)
                    metadata.update({
                        "completed_images": int(completion.sum()),
                        "last_checkpoint_utc": utc_now(),
                        "target_images_accessed": True,
                    })
                    atomic_json(metadata_path, metadata)
        memmap.flush()
        np.save(completion_path, completion, allow_pickle=False)
        del memmap

    assert completion.all()
    embeddings = np.load(embedding_path, mmap_mode="r", allow_pickle=False)
    assert embeddings.shape == expected_shape
    assert np.isfinite(embeddings).all()
    norms = np.linalg.norm(embeddings, axis=1)
    assert np.allclose(norms, 1.0, atol=1e-5)
    with metadata_path.open("r", encoding="utf-8") as handle:
        metadata = json.load(handle)
    metadata.update({
        "status": "COMPLETE",
        "completed_images": len(frame),
        "completed_utc": utc_now(),
        "embedding_sha256": sha256_file(embedding_path),
        "completion_sha256": sha256_file(completion_path),
        "image_id_sha256": sha256_file(image_id_path),
        "minimum_l2_norm": float(norms.min()),
        "maximum_l2_norm": float(norms.max()),
        "target_images_accessed": True,
        "target_sealed_labels_accessed": False,
        "target_performance_observed": False,
    })
    atomic_json(metadata_path, metadata)
    return embeddings, embedding_path, metadata_path


target_assets = {}
for target_name, frame in target_manifests.items():
    embeddings, embedding_path, metadata_path = load_or_extract_target_embeddings(target_name, frame)
    logits = raw_axis_logit(embeddings)
    probabilities = sigmoid(logits)
    score_table = frame[["dataset", "image_id", "canonical_sha256"]].copy()
    score_table["frozen_axis_logit"] = logits
    score_table["frozen_axis_probability"] = probabilities
    score_path = SCORE_ROOT / f"{target_name}_Frozen_DeepDRiD_Axis_Image_Scores_v0.1.csv"
    score_table.to_csv(score_path, index=False)
    target_assets[target_name] = {
        "frame": frame,
        "embeddings": embeddings,
        "logits": logits,
        "probabilities": probabilities,
        "embedding_path": embedding_path,
        "metadata_path": metadata_path,
        "score_path": score_path,
    }

print("\n================ TARGET EMBEDDINGS COMPLETE ================")
print("Maximum frozen-axis mathematical equivalence error:", maximum_axis_equivalence_error)
print("Target images accessed:", runtime_state["target_images_accessed"])
print("Target sealed-label files accessed: False")
print("Target performance observed: False")
display(pd.DataFrame([
    {
        "target": name,
        "images": len(asset["frame"]),
        "mean_probability": float(np.mean(asset["probabilities"])),
        "labels_used": False,
    }
    for name, asset in target_assets.items()
]))


Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 141MB/s]


APTOS_2019: completed embeddings 0/1080


CPU extracting APTOS_2019 target embeddings:   0%|          | 0/68 [00:00<?, ?it/s]

IDRiD: completed embeddings 0/102


CPU extracting IDRiD target embeddings:   0%|          | 0/7 [00:00<?, ?it/s]


================ TARGET EMBEDDINGS COMPLETE ================
Maximum frozen-axis mathematical equivalence error: 2.3092638912203256e-14
Target images accessed: True
Target sealed-label files accessed: False
Target performance observed: False


,target,images,mean_probability,labels_used
0,APTOS_2019,1080,0.477745,False
1,IDRiD,102,0.770950,False


In [3]:
#@title 05B-2. Compute frozen support, nuisance, confidence, ATC, MMD and OT components

from scipy.optimize import linear_sum_assignment
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler


def deterministic_indices(ids, count, namespace):
    order = sorted(
        range(len(ids)),
        key=lambda index: deterministic_rank(str(ids[index]), namespace),
    )
    return np.asarray(order[:min(count, len(order))], dtype=np.int64)


def cosine_knn_support(source_matrix, target_matrix, k=5):
    source = np.asarray(source_matrix, dtype=np.float64)
    target = np.asarray(target_matrix, dtype=np.float64)
    source_similarity = source @ source.T
    np.fill_diagonal(source_similarity, -np.inf)
    source_k_similarity = np.partition(source_similarity, -k, axis=1)[:, -k]
    source_distances = 1.0 - source_k_similarity
    target_similarity = target @ source.T
    target_k_similarity = np.partition(target_similarity, -k, axis=1)[:, -k]
    target_distances = 1.0 - target_k_similarity
    threshold_95 = float(np.quantile(source_distances, 0.95))
    threshold_99 = float(np.quantile(source_distances, 0.99))
    return {
        "source_distances": source_distances,
        "target_distances": target_distances,
        "source_q95": threshold_95,
        "source_q99": threshold_99,
        "support_fraction": float(np.mean(target_distances <= threshold_95)),
        "beyond_q99_fraction": float(np.mean(target_distances > threshold_99)),
    }


def pairwise_squared_distance(x, y):
    x = np.asarray(x, dtype=np.float64)
    y = np.asarray(y, dtype=np.float64)
    return np.maximum(
        np.sum(x * x, axis=1)[:, None] + np.sum(y * y, axis=1)[None, :] - 2.0 * x @ y.T,
        0.0,
    )


def rbf_mmd_squared(source, target):
    source_dist = pairwise_squared_distance(source, source)
    positive = source_dist[np.triu_indices_from(source_dist, k=1)]
    positive = positive[positive > 0]
    bandwidth = float(np.median(positive)) if len(positive) else 1.0
    bandwidth = max(bandwidth, 1e-12)
    k_xx = np.exp(-source_dist / (2.0 * bandwidth))
    k_yy = np.exp(-pairwise_squared_distance(target, target) / (2.0 * bandwidth))
    k_xy = np.exp(-pairwise_squared_distance(source, target) / (2.0 * bandwidth))
    return float(max(0.0, k_xx.mean() + k_yy.mean() - 2.0 * k_xy.mean())), bandwidth


def optimal_assignment_cosine_cost(source, target):
    cost = np.clip(1.0 - np.asarray(source, dtype=np.float64) @ np.asarray(target, dtype=np.float64).T, 0.0, 2.0)
    rows, columns = linear_sum_assignment(cost)
    return float(cost[rows, columns].mean())


def domain_classifier_auc(source, target):
    source = np.asarray(source, dtype=np.float64)
    target = np.asarray(target, dtype=np.float64)
    components = min(32, source.shape[0] - 1, source.shape[1])
    pca = PCA(n_components=components, svd_solver="randomized", random_state=RANDOM_SEED)
    pca.fit(source)
    features = np.vstack([pca.transform(source), pca.transform(target)])
    labels = np.concatenate([np.zeros(len(source), dtype=int), np.ones(len(target), dtype=int)])
    splitter = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)
    probabilities = np.full(len(labels), np.nan, dtype=np.float64)
    for train_indices, test_indices in splitter.split(features, labels):
        probe = Pipeline([
            ("scaler", StandardScaler()),
            ("classifier", LogisticRegression(
                C=1.0, class_weight="balanced", solver="liblinear",
                max_iter=5000, random_state=RANDOM_SEED,
            )),
        ])
        probe.fit(features[train_indices], labels[train_indices])
        probabilities[test_indices] = probe.predict_proba(features[test_indices])[:, 1]
    assert np.isfinite(probabilities).all()
    return float(roc_auc_score(labels, probabilities)), float(pca.explained_variance_ratio_.sum())


def binary_entropy(probabilities):
    probabilities = np.clip(np.asarray(probabilities, dtype=np.float64), 1e-12, 1.0 - 1e-12)
    return -(probabilities * np.log(probabilities) + (1.0 - probabilities) * np.log(1.0 - probabilities))


source_development_embeddings = np.asarray(
    source_eye_embeddings[source_development_mask], dtype=np.float32
)
source_development_ids = source_eye_table.loc[source_development_mask, "eye_unit_id"].astype(str).to_numpy()
source_validation = source_eye_table.loc[source_validation_mask].reset_index(drop=True).copy()
validation_confidence = np.maximum(
    source_validation["probability"].to_numpy(),
    1.0 - source_validation["probability"].to_numpy(),
)
validation_prediction = (source_validation["probability"].to_numpy() >= 0.5).astype(int)
validation_error_rate = float(np.mean(validation_prediction != source_validation["label"].to_numpy()))
atc_confidence_threshold = float(np.quantile(validation_confidence, validation_error_rate))

component_assets = {}
for target_name, asset in target_assets.items():
    target_embeddings = np.asarray(asset["embeddings"], dtype=np.float32)
    target_ids = asset["frame"]["image_id"].astype(str).to_numpy()
    support = cosine_knn_support(
        source_development_embeddings, target_embeddings, k=K_NEIGHBOURS
    )

    mmd_count = min(MMD_MAX_RECORDS, len(source_development_embeddings), len(target_embeddings))
    source_mmd_indices = deterministic_indices(source_development_ids, mmd_count, f"{target_name}|MMD|SOURCE")
    target_mmd_indices = deterministic_indices(target_ids, mmd_count, f"{target_name}|MMD|TARGET")
    mmd_squared, mmd_bandwidth = rbf_mmd_squared(
        source_development_embeddings[source_mmd_indices], target_embeddings[target_mmd_indices]
    )

    ot_count = min(OT_RECORDS, len(source_development_embeddings), len(target_embeddings))
    source_ot_indices = deterministic_indices(source_development_ids, ot_count, f"{target_name}|OT|SOURCE")
    target_ot_indices = deterministic_indices(target_ids, ot_count, f"{target_name}|OT|TARGET")
    ot_cost = optimal_assignment_cosine_cost(
        source_development_embeddings[source_ot_indices], target_embeddings[target_ot_indices]
    )
    domain_auc, pca_variance = domain_classifier_auc(
        source_development_embeddings, target_embeddings
    )

    target_confidence = np.maximum(asset["probabilities"], 1.0 - asset["probabilities"])
    baseline = {
        "mean_confidence": float(np.mean(target_confidence)),
        "mean_binary_entropy_nats": float(np.mean(binary_entropy(asset["probabilities"]))),
        "ATC_source_error_rate": validation_error_rate,
        "ATC_fixed_confidence_threshold": atc_confidence_threshold,
        "ATC_estimated_target_accuracy": float(np.mean(target_confidence >= atc_confidence_threshold)),
        "RBF_MMD_squared": mmd_squared,
        "RBF_MMD_bandwidth_source_median_squared_distance": mmd_bandwidth,
        "optimal_assignment_cosine_cost": ot_cost,
        "domain_classifier_auc": domain_auc,
        "domain_PCA_explained_variance": pca_variance,
        "source_validation_eye_auc": 0.924646,
        "source_validation_eye_auc_ci_95": [0.876887, 0.960957],
    }
    component_assets[target_name] = {
        "support": support,
        "baselines": baseline,
    }

support_rows = []
for target_name, values in component_assets.items():
    support_rows.append({
        "target": target_name,
        "support_fraction": values["support"]["support_fraction"],
        "beyond_source_q99_fraction": values["support"]["beyond_q99_fraction"],
        "domain_classifier_auc": values["baselines"]["domain_classifier_auc"],
        "RBF_MMD_squared": values["baselines"]["RBF_MMD_squared"],
        "optimal_assignment_cosine_cost": values["baselines"]["optimal_assignment_cosine_cost"],
        "ATC_estimated_target_accuracy": values["baselines"]["ATC_estimated_target_accuracy"],
    })
component_table = pd.DataFrame(support_rows)
component_table.to_csv(SCORE_ROOT / "Stage5B_LabelFree_Geometry_And_Baseline_Components_v0.1.csv", index=False)
display(component_table)
print("Target sealed-label files accessed: False")
print("Target performance observed: False")


,target,support_fraction,beyond_source_q99_fraction,domain_classifier_auc,RBF_MMD_squared,optimal_assignment_cosine_cost,ATC_estimated_target_accuracy
0,APTOS_2019,0.137963,0.579630,0.991418,0.193725,0.241159,0.933333
1,IDRiD,0.009804,0.617647,0.997745,0.313732,0.257977,0.960784


Target sealed-label files accessed: False
Target performance observed: False


In [4]:
#@title 05B-3. Frozen task-evidence response signatures and bootstrap uncertainty

# ============================================================
# 1. Fixed image transformations
# ============================================================

TRANSFORM_NAMES = list(TRANSFORMATION_SPEC.keys())

def apply_frozen_transform(image, transform_name, image_id):
    image = image.convert("RGB")
    assert image.size == (768, 768)
    if transform_name == "RESOLUTION_192":
        return image.resize((192, 192), Image.Resampling.BILINEAR).resize((768, 768), Image.Resampling.BICUBIC)
    if transform_name == "GAUSSIAN_BLUR_R2":
        return image.filter(ImageFilter.GaussianBlur(radius=2.0))
    if transform_name == "GAUSSIAN_NOISE_S004":
        seed = int(deterministic_rank(str(image_id), transform_name)[:16], 16) % (2**32)
        rng = np.random.default_rng(seed)
        array = np.asarray(image, dtype=np.float32) / 255.0
        noisy = np.clip(array + rng.normal(0.0, 0.04, size=array.shape), 0.0, 1.0)
        return Image.fromarray(np.round(noisy * 255.0).astype(np.uint8), mode="RGB")
    if transform_name == "CONTRAST_065":
        return ImageEnhance.Contrast(image).enhance(0.65)
    if transform_name == "SATURATION_050":
        return ImageEnhance.Color(image).enhance(0.50)
    if transform_name == "CENTRAL_CROP_080":
        margin = int(round(768 * 0.10))
        return image.crop((margin, margin, 768 - margin, 768 - margin)).resize((768, 768), Image.Resampling.BICUBIC)
    if transform_name == "CENTRAL_OCCLUSION_R015":
        output = image.copy()
        medians = tuple(int(value) for value in np.median(np.asarray(image), axis=(0, 1)))
        radius = int(round(768 * 0.15))
        center = 768 // 2
        ImageDraw.Draw(output).ellipse(
            (center - radius, center - radius, center + radius, center + radius),
            fill=medians,
        )
        return output
    if transform_name == "MEDIAN_FILTER_5":
        return image.filter(ImageFilter.MedianFilter(size=5))
    if transform_name == "JPEG_Q40":
        buffer = BytesIO()
        image.save(buffer, format="JPEG", quality=40, subsampling=2, optimize=False, progressive=False)
        buffer.seek(0)
        with Image.open(buffer) as decoded:
            decoded.load()
            return decoded.convert("RGB")
    raise KeyError(transform_name)


# ============================================================
# 2. Deterministic source-eye and target-image anchors
# ============================================================

validation_eye_ids = source_validation["eye_unit_id"].astype(str).to_numpy()
selected_source_eye_ids = []
for label in [0, 1]:
    candidates = source_validation[source_validation["label"].eq(label)]["eye_unit_id"].astype(str).tolist()
    candidates.sort(key=lambda value: deterministic_rank(value, f"SOURCE_ANCHOR_LABEL_{label}"))
    selected_source_eye_ids.extend(candidates[:SOURCE_ANCHOR_EYES_PER_CLASS])
selected_source_eye_ids = set(selected_source_eye_ids)
source_anchor_rows = source_manifest[source_manifest["eye_unit_id"].isin(selected_source_eye_ids)].copy()
assert source_anchor_rows["eye_unit_id"].nunique() == 2 * SOURCE_ANCHOR_EYES_PER_CLASS

target_anchor_rows = {}
for target_name, asset in target_assets.items():
    frame = asset["frame"].copy()
    frame["rank"] = frame["image_id"].astype(str).map(
        lambda value: deterministic_rank(value, f"{target_name}|TRANSFORM_ANCHOR")
    )
    target_anchor_rows[target_name] = frame.sort_values("rank").head(TRANSFORM_ANCHORS_PER_TARGET).drop(columns="rank")


# ============================================================
# 3. Resumable transformed logits
# ============================================================

transform_checkpoint_path = SCORE_ROOT / "Stage5B_Frozen_Transformation_Image_Logits_v0.1.csv"
completed_keys = set()
if transform_checkpoint_path.is_file():
    transform_image_table = pd.read_csv(transform_checkpoint_path)
    completed_keys = set(
        transform_image_table["record_key"].astype(str).tolist()
    )
else:
    transform_image_table = pd.DataFrame(columns=[
        "record_key", "dataset", "group_id", "image_id", "transform_name", "transformed_logit"
    ])

tasks = []
for row in source_anchor_rows.itertuples(index=False):
    for transform_name in TRANSFORM_NAMES:
        key = f"DeepDRiD|{row.eye_unit_id}|{row.image_id}|{transform_name}"
        if key not in completed_keys:
            tasks.append((key, "DeepDRiD", str(row.eye_unit_id), str(row.image_id), str(row.canonical_image_path), transform_name))
for target_name, frame in target_anchor_rows.items():
    for row in frame.itertuples(index=False):
        for transform_name in TRANSFORM_NAMES:
            key = f"{target_name}|{row.image_id}|{row.image_id}|{transform_name}"
            if key not in completed_keys:
                tasks.append((key, target_name, str(row.image_id), str(row.image_id), str(row.canonical_image_path), transform_name))

print(f"Transformed images remaining: {len(tasks)}")
new_rows = []
with torch.inference_mode():
    for start in tqdm(range(0, len(tasks), BATCH_SIZE), desc="CPU frozen evidence-response transforms"):
        batch_tasks = tasks[start:start + BATCH_SIZE]
        tensors = []
        for key, dataset_name, group_id, image_id, image_path, transform_name in batch_tasks:
            with Image.open(image_path) as image:
                image.load()
                transformed = apply_frozen_transform(image, transform_name, image_id)
                tensors.append(inference_transform(transformed))
        feature_batch = F.normalize(model(torch.stack(tensors).to(device)), p=2, dim=1)
        logits = raw_axis_logit(feature_batch.cpu().numpy().astype(np.float32))
        for task, logit in zip(batch_tasks, logits):
            key, dataset_name, group_id, image_id, image_path, transform_name = task
            new_rows.append({
                "record_key": key,
                "dataset": dataset_name,
                "group_id": group_id,
                "image_id": image_id,
                "transform_name": transform_name,
                "transformed_logit": float(logit),
            })
        if len(new_rows) >= 10 * BATCH_SIZE:
            transform_image_table = pd.concat([transform_image_table, pd.DataFrame(new_rows)], ignore_index=True)
            transform_image_table.to_csv(transform_checkpoint_path, index=False)
            new_rows = []
if new_rows:
    transform_image_table = pd.concat([transform_image_table, pd.DataFrame(new_rows)], ignore_index=True)
transform_image_table = transform_image_table.drop_duplicates("record_key", keep="last")
transform_image_table.to_csv(transform_checkpoint_path, index=False)

expected_transform_records = (
    len(source_anchor_rows) + sum(len(frame) for frame in target_anchor_rows.values())
) * len(TRANSFORM_NAMES)
assert len(transform_image_table) == expected_transform_records


# ============================================================
# 4. Aggregate source/target evidence responses
# ============================================================

source_base_by_image = source_manifest.set_index(source_manifest["image_id"].astype(str))["axis_logit"].to_dict()
target_base_by_image = {
    name: dict(zip(asset["frame"]["image_id"].astype(str), asset["logits"]))
    for name, asset in target_assets.items()
}
source_validation_logit_iqr = float(
    np.quantile(source_validation["axis_logit"], 0.75) - np.quantile(source_validation["axis_logit"], 0.25)
)
assert source_validation_logit_iqr > 0

source_transformed = transform_image_table[transform_image_table["dataset"].eq("DeepDRiD")].copy()
source_group_response_rows = []
for (group_id, transform_name), group in source_transformed.groupby(["group_id", "transform_name"]):
    base_logit = float(np.mean([source_base_by_image[str(image_id)] for image_id in group["image_id"]]))
    transformed_logit = float(group["transformed_logit"].mean())
    source_group_response_rows.append({
        "dataset": "DeepDRiD", "group_id": group_id, "transform_name": transform_name,
        "base_logit": base_logit, "transformed_logit": transformed_logit,
        "normalised_absolute_logit_change": abs(transformed_logit - base_logit) / source_validation_logit_iqr,
        "prediction_flip": int((base_logit >= 0) != (transformed_logit >= 0)),
    })
source_group_responses = pd.DataFrame(source_group_response_rows)
source_response_summary = source_group_responses.groupby("transform_name", as_index=False).agg(
    source_median_response=("normalised_absolute_logit_change", "median"),
    source_flip_rate=("prediction_flip", "mean"),
)
source_response_map = source_response_summary.set_index("transform_name")["source_median_response"].to_dict()

evidence_assets = {}
all_response_rows = [
    source_response_summary.assign(dataset="DeepDRiD").rename(columns={
        "source_median_response": "median_response", "source_flip_rate": "flip_rate"
    })[["dataset", "transform_name", "median_response", "flip_rate"]]
]
for target_name in target_assets:
    target_transformed = transform_image_table[transform_image_table["dataset"].eq(target_name)].copy()
    rows = []
    for record in target_transformed.itertuples(index=False):
        base_logit = float(target_base_by_image[target_name][str(record.image_id)])
        rows.append({
            "dataset": target_name,
            "group_id": str(record.group_id),
            "transform_name": str(record.transform_name),
            "base_logit": base_logit,
            "transformed_logit": float(record.transformed_logit),
            "normalised_absolute_logit_change": abs(float(record.transformed_logit) - base_logit) / source_validation_logit_iqr,
            "prediction_flip": int((base_logit >= 0) != (float(record.transformed_logit) >= 0)),
        })
    group_responses = pd.DataFrame(rows)
    summary = group_responses.groupby("transform_name", as_index=False).agg(
        target_median_response=("normalised_absolute_logit_change", "median"),
        target_flip_rate=("prediction_flip", "mean"),
    )
    summary["source_median_response"] = summary["transform_name"].map(source_response_map)
    summary["absolute_log_response_ratio"] = np.abs(np.log(
        (summary["target_median_response"] + 1e-6) /
        (summary["source_median_response"] + 1e-6)
    ))
    evidence_compatibility = float(np.exp(-summary["absolute_log_response_ratio"].mean()))
    evidence_assets[target_name] = {
        "group_responses": group_responses,
        "summary": summary,
        "evidence_compatibility": evidence_compatibility,
    }
    all_response_rows.append(summary.assign(dataset=target_name).rename(columns={
        "target_median_response": "median_response", "target_flip_rate": "flip_rate"
    })[["dataset", "transform_name", "median_response", "flip_rate"]])

transformation_summary_table = pd.concat(all_response_rows, ignore_index=True)
transformation_summary_path = SCORE_ROOT / "Stage5B_Frozen_Transformation_Response_Summary_v0.1.csv"
transformation_summary_table.to_csv(transformation_summary_path, index=False)


# ============================================================
# 5. Deterministic target bootstrap uncertainty
# ============================================================

def quantile_interval(values):
    return [float(np.quantile(values, 0.025)), float(np.quantile(values, 0.975))]

uncertainty_assets = {}
for target_index, target_name in enumerate(target_assets):
    rng = np.random.default_rng(RANDOM_SEED + 1000 + target_index)
    support_distances = component_assets[target_name]["support"]["target_distances"]
    support_threshold = component_assets[target_name]["support"]["source_q95"]
    support_bootstrap = np.empty(N_BOOTSTRAP, dtype=np.float64)
    evidence_bootstrap = np.empty(N_BOOTSTRAP, dtype=np.float64)
    response_table = evidence_assets[target_name]["group_responses"]
    group_ids = sorted(response_table["group_id"].unique())
    source_response_vector = source_response_summary.set_index("transform_name").loc[TRANSFORM_NAMES, "source_median_response"].to_numpy()
    for bootstrap_index in range(N_BOOTSTRAP):
        sampled_support = rng.integers(0, len(support_distances), size=len(support_distances))
        support_bootstrap[bootstrap_index] = np.mean(support_distances[sampled_support] <= support_threshold)
        sampled_groups = rng.choice(group_ids, size=len(group_ids), replace=True)
        sampled_rows = pd.concat([
            response_table[response_table["group_id"].eq(group_id)]
            for group_id in sampled_groups
        ], ignore_index=True)
        target_response_vector = (
            sampled_rows.groupby("transform_name")["normalised_absolute_logit_change"]
            .median().reindex(TRANSFORM_NAMES).to_numpy()
        )
        deviations = np.abs(np.log((target_response_vector + 1e-6) / (source_response_vector + 1e-6)))
        evidence_bootstrap[bootstrap_index] = np.exp(-np.mean(deviations))
    uncertainty_assets[target_name] = {
        "support_fraction_ci_95": quantile_interval(support_bootstrap),
        "evidence_compatibility_ci_95": quantile_interval(evidence_bootstrap),
        "bootstrap_replicates": N_BOOTSTRAP,
        "bootstrap_unit": "target image; fixed source reference",
    }

display(transformation_summary_table)
display(pd.DataFrame([
    {
        "target": name,
        "evidence_compatibility": evidence_assets[name]["evidence_compatibility"],
        "support_ci_95": uncertainty_assets[name]["support_fraction_ci_95"],
        "evidence_ci_95": uncertainty_assets[name]["evidence_compatibility_ci_95"],
    }
    for name in target_assets
]))
print("Target sealed-label files accessed: False")
print("Target performance observed: False")


Transformed images remaining: 2304


CPU frozen evidence-response transforms:   0%|          | 0/144 [00:00<?, ?it/s]

/tmp/ipykernel_3231/4100237390.py:21: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  return Image.fromarray(np.round(noisy * 255.0).astype(np.uint8), mode="RGB")
/tmp/ipykernel_3231/4100237390.py:127: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  transform_image_table = pd.concat([transform_image_table, pd.DataFrame(new_rows)], ignore_index=True)


,dataset,transform_name,median_response,flip_rate
0,DeepDRiD,CENTRAL_CROP_080,0.434747,0.187500
1,DeepDRiD,CENTRAL_OCCLUSION_R015,0.876177,0.406250
2,DeepDRiD,CONTRAST_065,0.100007,0.078125
3,DeepDRiD,GAUSSIAN_BLUR_R2,0.147121,0.203125
4,DeepDRiD,GAUSSIAN_NOISE_S004,0.242411,0.234375
5,DeepDRiD,JPEG_Q40,0.229907,0.140625
6,DeepDRiD,MEDIAN_FILTER_5,0.077289,0.125000
7,DeepDRiD,RESOLUTION_192,0.156459,0.187500
8,DeepDRiD,SATURATION_050,0.138797,0.109375
9,APTOS_2019,CENTRAL_CROP_080,1.105256,0.390625


,target,evidence_compatibility,support_ci_95,evidence_ci_95
0,APTOS_2019,0.449226,"[0.11710648148148149, 0.16018518518518518]","[0.3980444397733286, 0.5328191867091366]"
1,IDRiD,0.571785,"[0.0, 0.029411764705882353]","[0.5142214642555425, 0.6359589560553754]"


Target sealed-label files accessed: False
Target performance observed: False


In [5]:
#@title 05B-4. Freeze edge predictions, intervention recommendations, PDFs, and integrity manifest

import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages


EDGE_IDS = {"APTOS_2019": "DDR_APTOS_v0.1", "IDRiD": "DDR_IDRiD_v0.1"}
FAILURE_MODE_BY_TRANSFORM = {
    "RESOLUTION_192": "RESOLUTION_FREQUENCY_MISMATCH",
    "GAUSSIAN_BLUR_R2": "BLUR_NOISE_SENSITIVITY_MISMATCH",
    "GAUSSIAN_NOISE_S004": "BLUR_NOISE_SENSITIVITY_MISMATCH",
    "CONTRAST_065": "COLOUR_CONTRAST_MISMATCH",
    "SATURATION_050": "COLOUR_CONTRAST_MISMATCH",
    "CENTRAL_CROP_080": "FIELD_OF_VIEW_SUPPORT_MISMATCH",
    "CENTRAL_OCCLUSION_R015": "FIELD_OF_VIEW_SUPPORT_MISMATCH",
    "MEDIAN_FILTER_5": "STRUCTURE_TEXTURE_SUPPORT_MISMATCH",
    "JPEG_Q40": "COMPRESSION_ACQUISITION_MISMATCH",
}
INTERVENTION_BY_FAILURE = {
    "RESOLUTION_FREQUENCY_MISMATCH": "VERIFY_RESOLUTION_AND_USE_PRE_SPECIFIED_RESOLUTION_HARMONISATION_OR_REACQUIRE",
    "BLUR_NOISE_SENSITIVITY_MISMATCH": "QUALITY_TRIAGE_AND_REACQUIRE_LOW_QUALITY_IMAGES",
    "COLOUR_CONTRAST_MISMATCH": "APPLY_ONLY_PRE_SPECIFIED_COLOUR_ILLUMINATION_NORMALISATION",
    "FIELD_OF_VIEW_SUPPORT_MISMATCH": "HARMONISE_FIELD_OF_VIEW_OR_REACQUIRE",
    "STRUCTURE_TEXTURE_SUPPORT_MISMATCH": "OBTAIN_TARGET_LABELS_BEFORE_ANY_TEXTURE_SPECIFIC_ADAPTATION",
    "COMPRESSION_ACQUISITION_MISMATCH": "USE_LOSSLESS_OR_HIGHER_QUALITY_SOURCE_IMAGES",
}


def decision_for_target(target_name):
    support = component_assets[target_name]["support"]
    baseline = component_assets[target_name]["baselines"]
    evidence = evidence_assets[target_name]
    uncertainty = uncertainty_assets[target_name]
    reasons = []
    if support["support_fraction"] < DECISION_THRESHOLDS["abstain_if_support_fraction_below"]:
        reasons.append("LOW_SOURCE_SUPPORT")
    if support["beyond_q99_fraction"] > DECISION_THRESHOLDS["abstain_if_fraction_beyond_source_q99_above"]:
        reasons.append("MAJORITY_BEYOND_SOURCE_Q99")
    if evidence["evidence_compatibility"] < DECISION_THRESHOLDS["abstain_if_evidence_compatibility_below"]:
        reasons.append("EVIDENCE_RESPONSE_OUT_OF_SUPPORT")
    abstain = bool(reasons)

    if abstain:
        transfer_risk = "NOT_CALIBRATED_OUT_OF_SUPPORT"
    elif (
        support["support_fraction"] < DECISION_THRESHOLDS["high_risk_if_support_fraction_below"] or
        evidence["evidence_compatibility"] < DECISION_THRESHOLDS["high_risk_if_evidence_compatibility_below"] or
        baseline["domain_classifier_auc"] >= DECISION_THRESHOLDS["high_risk_if_domain_auc_at_or_above"]
    ):
        transfer_risk = "HIGH"
    elif (
        support["support_fraction"] < DECISION_THRESHOLDS["moderate_risk_if_support_fraction_below"] or
        evidence["evidence_compatibility"] < DECISION_THRESHOLDS["moderate_risk_if_evidence_compatibility_below"] or
        baseline["domain_classifier_auc"] >= DECISION_THRESHOLDS["moderate_risk_if_domain_auc_at_or_above"]
    ):
        transfer_risk = "MODERATE"
    else:
        transfer_risk = "LOW"

    response_summary = evidence["summary"].copy()
    dominant_transform = str(
        response_summary.sort_values("absolute_log_response_ratio", ascending=False).iloc[0]["transform_name"]
    )
    dominant_failure_mode = FAILURE_MODE_BY_TRANSFORM[dominant_transform]
    if abstain:
        minimal_intervention = "OBTAIN_TARGET_LABELS_BEFORE_DEPLOYMENT_OR_ADAPTATION"
    elif transfer_risk == "LOW":
        minimal_intervention = "NONE_CONTINUE_TO_SEPARATELY_GOVERNED_UNSEALING"
    else:
        minimal_intervention = INTERVENTION_BY_FAILURE[dominant_failure_mode]

    narrative = (
        f"Before target-label access, the frozen DeepDRiD axis places {target_name} in the "
        f"{transfer_risk} transfer-risk category. Source-support fraction is "
        f"{support['support_fraction']:.3f} and evidence-response compatibility is "
        f"{evidence['evidence_compatibility']:.3f}; the label-free domain-classifier AUC is "
        f"{baseline['domain_classifier_auc']:.3f}. The dominant pre-specified mismatch code is "
        f"{dominant_failure_mode}. Abstention is {abstain}. The recommended next action is "
        f"{minimal_intervention}. This is a two-edge prospective feasibility prediction, not a "
        "calibrated target AUC estimate or a validated general transfer predictor."
    )
    assert len(narrative.split()) <= 150
    return {
        "edge_id": EDGE_IDS[target_name],
        "source": "DeepDRiD",
        "target": target_name,
        "eligibility": "AUTHORISED_BY_SEALED_STAGE5A_SOURCE_GATE",
        "component_scores": {
            "source_support_fraction": support["support_fraction"],
            "fraction_beyond_source_q99": support["beyond_q99_fraction"],
            "source_knn_distance_q95": support["source_q95"],
            "source_knn_distance_q99": support["source_q99"],
            "evidence_response_compatibility": evidence["evidence_compatibility"],
            "domain_classifier_auc": baseline["domain_classifier_auc"],
            "RBF_MMD_squared": baseline["RBF_MMD_squared"],
            "optimal_assignment_cosine_cost": baseline["optimal_assignment_cosine_cost"],
        },
        "baselines": baseline,
        "uncertainty": uncertainty,
        "transfer_risk": transfer_risk,
        "dominant_transform": dominant_transform,
        "dominant_failure_mode": dominant_failure_mode,
        "minimal_intervention": minimal_intervention,
        "abstain": abstain,
        "abstention_reason_codes": reasons,
        "relative_statement": "NO_RELATIVE_ORDERING_PRECOMMITTED_FOR_N_EQUALS_2",
        "narrative": narrative,
        "target_labels_accessed": False,
        "target_performance_observed": False,
        "input_hashes": input_hashes,
        "output_created_utc": utc_now(),
    }


edge_records = [decision_for_target(target_name) for target_name in ["APTOS_2019", "IDRiD"]]
for record in edge_records:
    edge_path = FREEZE_ROOT / f"{record['edge_id']}_PreUnseal_Prediction_Record_v0.1.json"
    if edge_path.is_file():
        with edge_path.open("r", encoding="utf-8") as handle:
            existing = json.load(handle)
        assert sha256_json({key: value for key, value in existing.items() if key != "output_created_utc"}) == sha256_json(
            {key: value for key, value in record.items() if key != "output_created_utc"}
        )
        record = existing
    else:
        atomic_json(edge_path, record)

    pdf_path = FREEZE_ROOT / f"{record['edge_id']}_PreUnseal_Decision_Sheet_v0.1.pdf"
    if not pdf_path.is_file():
        with PdfPages(pdf_path) as pdf:
            figure = plt.figure(figsize=(8.5, 11))
            figure.suptitle(f"Stage 5B Pre-Unseal Decision — {record['edge_id']}", fontsize=16, weight="bold", y=0.97)
            lines = [
                "SOURCE: DeepDRiD", f"TARGET: {record['target']}",
                f"TRANSFER RISK: {record['transfer_risk']}", f"ABSTAIN: {record['abstain']}",
                f"DOMINANT FAILURE MODE: {record['dominant_failure_mode']}",
                f"MINIMAL INTERVENTION: {record['minimal_intervention']}", "",
                "Frozen component values:",
            ]
            lines.extend([f"  {key}: {value}" for key, value in record["component_scores"].items()])
            lines.extend(["", "Uncertainty:"])
            lines.extend([f"  {key}: {value}" for key, value in record["uncertainty"].items()])
            lines.extend(["", "Prospective narrative:", record["narrative"], "", "Target labels accessed: False", "Target performance observed: False"])
            figure.text(0.08, 0.92, "\n".join(lines), va="top", ha="left", fontsize=9, wrap=True, family="monospace")
            pdf.savefig(figure, bbox_inches="tight")
            plt.close(figure)

summary_rows = []
for record in edge_records:
    summary_rows.append({
        "edge_id": record["edge_id"], "source": record["source"], "target": record["target"],
        "transfer_risk": record["transfer_risk"], "abstain": record["abstain"],
        "abstention_reason_codes": "|".join(record["abstention_reason_codes"]),
        "source_support_fraction": record["component_scores"]["source_support_fraction"],
        "evidence_response_compatibility": record["component_scores"]["evidence_response_compatibility"],
        "domain_classifier_auc": record["component_scores"]["domain_classifier_auc"],
        "dominant_failure_mode": record["dominant_failure_mode"],
        "minimal_intervention": record["minimal_intervention"],
        "target_labels_accessed": False, "target_performance_observed": False,
    })
edge_summary = pd.DataFrame(summary_rows)
edge_summary_path = FREEZE_ROOT / "Stage5B_PreUnseal_Edge_Prediction_Summary_v0.1.csv"
edge_summary.to_csv(edge_summary_path, index=False)

report_path = RESULT_ROOT / "Stage5B_Blind_LabelFree_Scoring_And_Prediction_Freeze_Report_v0.1.md"
report_lines = [
    "# Stage 5B — Blind Label-Free Scoring and Prediction Freeze", "",
    f"- Protocol seal: `{seal_payload['seal_sha256']}`",
    "- Target images accessed after seal: `True`",
    "- Target sealed-label files accessed: `False`",
    "- Target performance observed: `False`", "",
    "## Frozen edge decisions", "",
]
for record in edge_records:
    report_lines.extend([
        f"### {record['edge_id']}", "", record["narrative"], "",
        f"- Abstention reasons: `{record['abstention_reason_codes']}`", "",
    ])
report_path.write_text("\n".join(report_lines), encoding="utf-8")

environment_path = RESULT_ROOT / "Stage5B_Environment_v0.1.json"
atomic_json(environment_path, environment)

output_candidates = sorted([
    path for root in [SCORE_ROOT, FREEZE_ROOT, RESULT_ROOT]
    for path in root.iterdir()
    if (
        path.is_file()
        and path != FINAL_FREEZE_PATH
        and path != RUNTIME_STATE_PATH
        and "Integrity_Manifest" not in path.name
    )
], key=lambda path: str(path))
integrity_rows = [{
    "relative_path": str(path.relative_to(STAGE5B_ROOT)),
    "size_bytes": path.stat().st_size,
    "sha256": sha256_file(path),
} for path in output_candidates]
integrity_manifest = pd.DataFrame(integrity_rows)
integrity_manifest_path = FREEZE_ROOT / "Stage5B_PreUnseal_Output_Integrity_Manifest_v0.1.csv"
integrity_manifest.to_csv(integrity_manifest_path, index=False)

final_freeze_payload = {
    "stage": "Stage5B",
    "decision": "PREDICTIONS_FROZEN_ADVANCE_TO_SEPARATELY_GOVERNED_STAGE6_UNSEALING",
    "protocol_seal_sha256": seal_payload["seal_sha256"],
    "output_integrity_manifest_path": str(integrity_manifest_path),
    "output_integrity_manifest_sha256": sha256_file(integrity_manifest_path),
    "edge_prediction_record_sha256": {
        record["edge_id"]: sha256_file(FREEZE_ROOT / f"{record['edge_id']}_PreUnseal_Prediction_Record_v0.1.json")
        for record in edge_records
    },
    "target_images_accessed_after_protocol_seal": True,
    "target_sealed_label_files_accessed": False,
    "target_performance_observed": False,
    "source_target_transfer_performance_observed": False,
    "frozen_utc": utc_now(),
    "unsealing_authorised_inside_this_notebook": False,
}
final_freeze_payload["freeze_record_sha256"] = sha256_json(final_freeze_payload)

if FINAL_FREEZE_PATH.is_file():
    with FINAL_FREEZE_PATH.open("r", encoding="utf-8") as handle:
        existing_final = json.load(handle)
    assert existing_final["protocol_seal_sha256"] == seal_payload["seal_sha256"]
    assert existing_final["output_integrity_manifest_sha256"] == sha256_file(integrity_manifest_path)
    final_freeze_payload = existing_final
else:
    atomic_json(FINAL_FREEZE_PATH, final_freeze_payload)

runtime_state.update({
    "target_images_accessed": True,
    "target_sealed_label_files_accessed": False,
    "target_performance_observed": False,
    "source_target_transfer_performance_observed": False,
    "prediction_freeze_complete": True,
    "prediction_freeze_path": str(FINAL_FREEZE_PATH),
    "prediction_freeze_sha256": final_freeze_payload["freeze_record_sha256"],
    "last_updated_utc": utc_now(),
})
atomic_json(RUNTIME_STATE_PATH, runtime_state)

print("\n================ STAGE 5B PREDICTION FREEZE COMPLETE ================")
display(edge_summary)
print("\nFinal freeze record:", FINAL_FREEZE_PATH)
print("Freeze record hash:", final_freeze_payload["freeze_record_sha256"])
print("Target images accessed after seal: True")
print("Target sealed-label files accessed: False")
print("Target performance observed: False")
print("Source-target transfer performance observed: False")
print("\nSTOP HERE. Stage 6 unsealing is not authorised inside this notebook.")



================ STAGE 5B PREDICTION FREEZE COMPLETE ================


,edge_id,source,target,transfer_risk,abstain,abstention_reason_codes,source_support_fraction,evidence_response_compatibility,domain_classifier_auc,dominant_failure_mode,minimal_intervention,target_labels_accessed,target_performance_observed
0,DDR_APTOS_v0.1,DeepDRiD,APTOS_2019,NOT_CALIBRATED_OUT_OF_SUPPORT,True,LOW_SOURCE_SUPPORT|MAJORITY_BEYOND_SOURCE_Q99,0.137963,0.449226,0.991418,COLOUR_CONTRAST_MISMATCH,OBTAIN_TARGET_LABELS_BEFORE_DEPLOYMENT_OR_ADAP...,False,False
1,DDR_IDRiD_v0.1,DeepDRiD,IDRiD,NOT_CALIBRATED_OUT_OF_SUPPORT,True,LOW_SOURCE_SUPPORT|MAJORITY_BEYOND_SOURCE_Q99,0.009804,0.571785,0.997745,BLUR_NOISE_SENSITIVITY_MISMATCH,OBTAIN_TARGET_LABELS_BEFORE_DEPLOYMENT_OR_ADAP...,False,False



Final freeze record: /content/drive/MyDrive/Cross-Modal_Diagnostic_Observability/06_Data_Records/Retinal_DR/Prospective_Retinal_Blind_Test_v0.1/Stage5B_Label_Free_Target_Scoring_And_Prediction_Freeze_v0.1/02_Prediction_Freeze/Stage5B_Prediction_Freeze_Complete_v0.1.json
Freeze record hash: f263225b6e44c7fc3b7709533afd5aae8b04d47b7ec77dc8d0c5bac911e5f1b9
Target images accessed after seal: True
Target sealed-label files accessed: False
Target performance observed: False
Source-target transfer performance observed: False

STOP HERE. Stage 6 unsealing is not authorised inside this notebook.


## Stop boundary

A successful final cell means that the two edge prediction records and their hashes are frozen. Do not open target labels in this notebook. Stage 6 must be a separate, explicitly authorised unsealing workflow that first verifies the Stage 5B freeze record and integrity manifest.
